# فاز اول: رابطهٔ MoCA و ساختار انتورینال در سالمندان NACC

این دفترچه تحلیل بازتولیدپذیر داده‌های UDS و MRI را اجرا می‌کند. جمعیت تحلیلی با تعریف عملیاتی سن ۶۵ سال و بیشتر ساخته شده است. هر فایل ورودی در نسخهٔ حاضر یک ردیف به‌ازای هر NACCID دارد؛ چون شناسه/تاریخ مراجعه در فایل نیست، انتخاب Baseline زمانی ممکن نیست. کد، اتصال بر اساس شناسه، شمارش سطرها و افراد، کیفیت داده، آمار توصیفی و نمودارها را می‌سازد. سؤال تعدیل‌شده با سن و تحصیلات برای فاز مدل‌سازی بعدی است؛ خروجی این فاز توصیفی است.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "udslong_imputed.csv").exists())
subprocess.run([sys.executable, str(ROOT / "src" / "phase1_analysis.py")], check=True, cwd=ROOT)
results = json.loads((ROOT / "reports/phase1/phase1_results.json").read_text(encoding="utf-8"))
print(json.dumps(results["linkage"], ensure_ascii=False, indent=2))

## قاعدهٔ نمونه و اتصال

NACCID واحد تحلیل است. در فایل‌های ارائه‌شده، هر دو جدول ۱٬۲۳۷ شناسهٔ یکتا و بدون تکرار دارند؛ پس یک ردیف موجود برای هر فرد نگه داشته می‌شود. فایل‌ها فیلد visit/date ندارند و نمی‌توان ادعا کرد این ردیف‌ها Baseline یا از نظر زمانی هم‌زمان‌اند. سپس افراد با سن NACCAGE ≥ 65 برای جمعیت هدف انتخاب می‌شوند.

In [ ]:
print("طرح:", results["design"])
print("تطبیق:", results["linkage"])

## آمار توصیفی و گمشدگی

SD انحراف معیار نمونه است؛ IQR با چارک‌های خطی محاسبه می‌شود و چولگی، چولگی نمونهٔ تعدیل‌شده است. EDUC=99 نامعلوم و MOCATOTS خارج از ۰–۳۰ نامعتبر تلقی می‌شود. حجم صفر/منفی برای خلاصهٔ حجم قابل استفاده نیست. مقدار imputed موجود، جایگزین مشاهدهٔ خام نیست و روش جایگذاری در فایل‌ها مستند نشده است.

In [ ]:
for name, s in results["summary_statistics"].items():
    print(f"{name}: n={s['n']}, mean={s['mean']:.3f}, median={s['median']:.3f}, SD={s['sd']:.3f}, IQR={s['iqr']:.3f}, skew={s['skewness']:.3f}")
print("Missingness:")
for row in results["missingness"]: print(row)

## چهار نمودار اصلی و برداشت توصیفی

خروجی‌ها شامل هیستوگرام MOCATOTS، پراکنش حجم Entorhinal در برابر MOCATOTS با روند خطی ساده، boxplot نمره برحسب تشخیص، و heatmap همبستگی Pearson جفتی هستند. خط روند/همبستگی توصیفی‌اند و اثر سن و تحصیلات را کنترل نمی‌کنند؛ برای سؤال اصلی باید در فاز بعد رگرسیون چندمتغیره برازش و فروض آن بررسی شوند.

In [ ]:
for name in results["figure_files"]:
    print(name)

## تفسیر محتاطانه

مقادیر همبستگی جفتی در heatmap خام‌اند. تحلیل تعدیل‌شدهٔ موردنظر را می‌توان در فاز بعد به صورت MOCATOTS ~ حجم/ضخامت Entorhinal + NACCAGE + EDUC تعریف کرد؛ پیش از آن باید الگوی گمشدگی، اثر جایگذاری، کیفیت/واحدهای MRI، هم‌زمانی سنجش و شکل رابطه بررسی شوند. نتایج این زیرنمونه را نمی‌توان به جمعیت عمومی تعمیم داد و رابطهٔ مشاهده‌ای علیت را نشان نمی‌دهد.